# BluaDiagnostics x Care Plus — PoC Sprint 1

Este notebook demonstra:
- System prompt aplicado
- Memoria de sessao com 3+ turnos
- Function calling com pelo menos 1 tool


In [2]:
pip install anthropic python-dotenv

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [48]:
# O símbolo de percentagem garante que a instalação vai para o Kernel ativo do seu notebook
%pip install --user tiktoken requests python-dotenv

     |████████████████████████████████| 987 kB 3.8 MB/s eta 0:00:01
     |████████████████████████████████| 64 kB 9.0 MB/s  eta 0:00:01
     |████████████████████████████████| 288 kB 5.4 MB/s eta 0:00:01
     |████████████████████████████████| 299 kB 5.2 MB/s eta 0:00:01
     |████████████████████████████████| 131 kB 9.8 MB/s eta 0:00:01
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [59]:
import os
from ollama import Client
from dotenv import load_dotenv

# Carrega a chave do arquivo .env de forma invisível
load_dotenv()

# Configuração correta utilizando a API externa fornecida pelo professor
client = Client(
    host="https://ollama.com",
    headers={'Authorization': f"Bearer {os.getenv('OLLAMA_API_KEY')}"}
)

MODEL_NAME = "gpt-oss:120b"


In [104]:
import os
import tiktoken
import warnings
from urllib3.exceptions import NotOpenSSLWarning
from ollama import Client
from dotenv import load_dotenv

# Ocultar o aviso do Mac para deixar o notebook limpo
warnings.filterwarnings("ignore", category=NotOpenSSLWarning)

# Carrega as variáveis do ficheiro .env que está na mesma pasta
load_dotenv()

# Puxa a chave de forma segura
api_key = os.getenv('OLLAMA_API_KEY')

# Configuração Ollama Cloud
client = Client(
    host="https://ollama.com",
    headers={'Authorization': 'Bearer ' + api_key} if api_key else {}
)

MODEL_NAME = "gpt-oss:120b"

# Função isolada para contar tokens
def contar_tokens(texto):
    try:
        encoding = tiktoken.get_encoding("cl100k_base")
        return len(encoding.encode(texto)) if texto else 0
    except Exception:
        return 0

# A sua função LLM: Limite estrito de 400 tokens e temperatura 0.2
def llm(mensagens, tools=None, max_tokens=400, temperature=0.2): 
    try:
        return client.chat( 
            model=MODEL_NAME,
            messages=mensagens,
            tools=tools, 
            options={"num_predict": max_tokens, "temperature": temperature},
            stream=False
        )['message'] 
    
    except Exception as e:
        return {"role": "assistant", "content": f"⚠️ Erro de API: {e}"}

print("Estrutura Ollama configurada para 400 tokens!")

Estrutura Ollama configurada para 400 tokens!


In [98]:
import json

# CARREGAMENTO SEUS ARQUIVOS LOCAIS

# Carregar o System Prompt 
with open('../prompts/system_prompt.md', 'r', encoding='utf-8') as f:
    system_prompt = f.read()

# Carregar as Ferramentas (Arquivo JSON)
with open('../tools/tools_spec.json', "r", encoding="utf-8") as f:
    dados_json = json.load(f)
    tools_spec = dados_json.get("tools", dados_json)

print("Arquivos carregados com sucesso e prontos para uso!")

Arquivos carregados com sucesso e prontos para uso!


In [ ]:
# ==========================================
# FUNÇÕES DO SISTEMA (MOCKS PARA TESTE)
# ==========================================
# No futuro, você vai substituir os "returns" destas funções por conexões reais com seu banco de dados (Care Plus)

def consultar_historico_paciente(paciente_id, janela_meses=12):
    return json.dumps({"status": "sucesso", "dados": f"Paciente {paciente_id}: Hipertenso, sem alergias graves registradas nos últimos {janela_meses} meses."})

def verificar_interacoes_medicamentosas(medicamentos_em_uso, novo_medicamento):
    return json.dumps({"status": "alerta", "severidade": "leve", "recomendacao": f"Atenção ao combinar {novo_medicamento} com {medicamentos_em_uso}. Monitorar pressão arterial."})

def agendar_teleconsulta(paciente_id, especialidade, urgencia):
    return json.dumps({"status": "sucesso", "mensagem": f"Teleconsulta de {especialidade} agendada com urgência '{urgencia}'."})

def registrar_triagem_sintomas(paciente_id, sintomas_coletados, red_flag_detectada, proxima_acao, duracao="N/A", intensidade=0):
    return json.dumps({"status": "sucesso", "mensagem": "Dados da triagem salvos silenciosamente no sistema."})

def buscar_protocolo_clinico(sintoma_principal):
    return json.dumps({"status": "sucesso", "protocolo": f"Para {sintoma_principal}: Avaliar sinais vitais, questionar sobre febre e intensidade da dor."})

def consultar_rede_credenciada(cep_paciente, tipo_unidade):
    return json.dumps({"status": "sucesso", "unidades": f"Hospital Care Plus mais próximo do CEP {cep_paciente} operando normalmente."})

def solicitar_renovacao_receita(paciente_id, nome_medicamento):
    return json.dumps({"status": "sucesso", "mensagem": f"Solicitação de renovação do medicamento {nome_medicamento} enviada ao médico responsável."})

def verificar_cobertura_plano(paciente_id, nome_procedimento):
    return json.dumps({"status": "sucesso", "cobertura": f"O procedimento {nome_procedimento} possui cobertura total."})

def consultar_resultado_exame(paciente_id, tipo_exame):
    return json.dumps({"status": "sucesso", "resultado": f"O último {tipo_exame} apresentou resultados dentro da normalidade."})

def acionar_emergencia_samu(paciente_id, motivo_gravidade):
    return json.dumps({"status": "emergencia_acionada", "instrucao": f"Alerta máximo para {motivo_gravidade}. Oriente o paciente a ligar 192 imediatamente."})


# ==========================================
# MAPA DE FUNÇÕES (A PONTE COM O LLM)
# ==========================================
funcoes_disponiveis = {
    "consultar_historico_paciente": consultar_historico_paciente,
    "verificar_interacoes_medicamentosas": verificar_interacoes_medicamentosas,
    "agendar_teleconsulta": agendar_teleconsulta,
    "registrar_triagem_sintomas": registrar_triagem_sintomas,
    "buscar_protocolo_clinico": buscar_protocolo_clinico,
    "consultar_rede_credenciada": consultar_rede_credenciada,
    "solicitar_renovacao_receita": solicitar_renovacao_receita,
    "verificar_cobertura_plano": verificar_cobertura_plano,
    "consultar_resultado_exame": consultar_resultado_exame,
    "acionar_emergencia_samu": acionar_emergencia_samu
}

print("Ambiente, LLM e todas as 10 ferramentas médicas configuradas!")

✅ Ambiente, LLM e todas as 10 ferramentas médicas configuradas!


In [108]:
# ==========================================
# SIMULAÇÃO DE MÚLTIPLOS TURNOS COM MEMÓRIA
# ==========================================
historico_mensagens = [
    {"role": "system", "content": system_prompt}
]

# Cenário de teste focado nas ferramentas médicas mapeadas
interacoes_usuario = [
    "Olá, meu ID é CP-8842. Tenho sentido muita falta de ar e dor no peito desde a manhã. Queria saber se meu plano cobre um eletrocardiograma.", 
    "Entendi. Eu já tomo Losartana para pressão e me recomendaram tomar Sildenafila. Tem algum problema misturar?", 
    "Perfeito, obrigado pelo aviso. Pode agendar uma teleconsulta urgente com um cardiologista para mim então?"
]

for i, texto_usuario in enumerate(interacoes_usuario, 1):
    print(f"\n--- Turno {i} ---")
    
    # Exibe a contagem de tokens elegantemente ao lado da fala do usuário
    tokens_usuario = contar_tokens(texto_usuario)
    print(f"👤 Usuário [{tokens_usuario} tokens]: {texto_usuario}")
    
    historico_mensagens.append({"role": "user", "content": texto_usuario})
    resposta_llm = llm(mensagens=historico_mensagens, tools=tools_spec)
    
    # Gerencia a chamada de ferramenta de forma limpa
    if resposta_llm.get("tool_calls"):
        print(f"🤖 [Processando informações internamente...]")
        
        # 1. Salva a decisão do assistente de usar as ferramentas no histórico
        historico_mensagens.append(resposta_llm)
        
        # 2. Executa todas as ferramentas que o LLM pediu em paralelo
        for tool_call in resposta_llm["tool_calls"]:
            nome_funcao = tool_call["function"]["name"]
            argumentos = tool_call["function"]["arguments"]
            
            if nome_funcao in funcoes_disponiveis:
                # O usuário não vê o JSON das ferramentas rodando aqui
                resultado_funcao = funcoes_disponiveis[nome_funcao](**argumentos)
                
                # Anexa o resultado de cada ferramenta no histórico
                historico_mensagens.append({
                    "role": "tool",
                    "name": nome_funcao,
                    "content": resultado_funcao
                })
        
        # 3. Faz uma nova chamada para o LLM gerar a resposta final após ler os resultados
        resposta_llm = llm(mensagens=historico_mensagens, tools=tools_spec)
    
    # Exibe a resposta final e humana para o usuário
    conteudo_resposta = resposta_llm.get("content", "Sem conteúdo retornado.")
    print(f"🤖 Assistente: {conteudo_resposta}")
    
    # Salva a resposta final na memória para o próximo turno
    historico_mensagens.append({"role": "assistant", "content": conteudo_resposta})


--- Turno 1 ---
👤 Usuário [47 tokens]: Olá, meu ID é CP-8842. Tenho sentido muita falta de ar e dor no peito desde a manhã. Queria saber se meu plano cobre um eletrocardiograma.
🤖 [Processando informações internamente...]
🤖 Assistente: Olá, sinto muito que você esteja passando por isso. Falta de ar acompanhada de dor no peito pode ser sinal de algo sério que precisa de avaliação imediata.

⚠️ **Por favor, procure atendimento de urgência agora mesmo** – vá ao pronto‑socorro mais próximo ou ligue para o SAMU (192) sem demora.

Sua segurança é a prioridade. Se precisar de mais alguma orientação depois de receber atendimento, estou à disposição. Cuide‑se!

--- Turno 2 ---
👤 Usuário [29 tokens]: Entendi. Eu já tomo Losartana para pressão e me recomendaram tomar Sildenafila. Tem algum problema misturar?
🤖 [Processando informações internamente...]
🤖 Assistente: A Losartana e a Sildenafil podem ser usadas juntas, mas é importante ficar atento:

* **Interação:** A combinação pode causar uma qu